# 09 — Estimate and save the models

This notebook is self-contained. It rebuilds both model datasets, estimates the models, and writes portable CSV/JSON results to `ANAL/data/models`. Each major step prints only the input shape, retained observations, model status, or saved output.


## 1. Paths and input files


In [ ]:
from pathlib import Path
import json
import platform
from datetime import datetime, timezone

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "ANAL":
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / "ANAL").is_dir():
    raise RuntimeError("Start the notebook from the repository root or the ANAL directory.")

DATA_DIR = PROJECT_DIR / "ANAL" / "data"
FEATURE_DIR = DATA_DIR / "routing" / "features"
PANEL_PATH = DATA_DIR / "raster_quarter_panel_100m.parquet"
FIRMS_PATH = DATA_DIR / "firms_assigned_100m.geoparquet"
BIRTHS_BY_GROUP_PATH = DATA_DIR / "births_by_fachgruppe_100m.parquet"
RESULT_DIR = DATA_DIR / "models"
CACHE_DIR = DATA_DIR / "cache"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2016
END_YEAR = 2025
LAG_YEAR = 2015

print(f"Project: {PROJECT_DIR}")
print(f"Analysis period: {START_YEAR}–{END_YEAR}")
print(f"Model results: {RESULT_DIR}")


In [ ]:
input_files = pd.DataFrame([
    {"input": "cell-quarter panel", "path": PANEL_PATH, "exists": PANEL_PATH.exists()},
    {"input": "firm records", "path": FIRMS_PATH, "exists": FIRMS_PATH.exists()},
    {"input": "births by Fachgruppe", "path": BIRTHS_BY_GROUP_PATH, "exists": BIRTHS_BY_GROUP_PATH.exists()},
])
display(input_files)
if not input_files["exists"].all():
    raise FileNotFoundError("At least one main model input is missing.")


## 2. Check required files and columns


In [ ]:
def require_columns(path, required):
    if not path.exists():
        raise FileNotFoundError(path)
    available = set(pq.ParquetFile(path).schema_arrow.names)
    missing = sorted(set(required) - available)
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")

require_columns(PANEL_PATH, ["grid_id", "municipality_id", "year", "quarter", "period", "births", "active_firms_tminus1"])
require_columns(FIRMS_PATH, ["firm_id", "founding_date", "exit_date", "exit_observed", "Fachgruppe_ID", "Sparte_ID", "Sparte_Text"])

for year in range(LAG_YEAR, END_YEAR + 1):
    year_dir = FEATURE_DIR / str(year)
    require_columns(year_dir / "accessibility_potentials_100m.parquet", [
        "grid_id", "year", "quarter", "period", "own_cell_pop", "own_cell_firms",
        "pop_access_15min", "existing_firms_access_15min",
    ])
    require_columns(year_dir / "pedestrian_accessibility_quarter_100m.parquet", [
        "grid_id", "year", "quarter", "period", "own_cell_walk_pop", "own_cell_walk_firms",
        "walk_pop_10min", "walk_firms_10min", "walk_pt_routes_10min", "pt_ohne_haltestelle",
    ])
    require_columns(year_dir / "nearest_infrastructure_100m.parquet", ["grid_id", "year", "tt_motorway_exit_min"])

for year in range(START_YEAR, END_YEAR + 1):
    require_columns(FEATURE_DIR / str(year) / "firm_accessibility_quarter_100m.parquet", [
        "firm_id", "grid_id_100m", "Fachgruppe_ID", "year", "quarter", "period",
        "included_in_lagged_stock", "own_cell_pop", "own_cell_firms",
        "own_cell_same_fachgruppe_firms", "pop_access_15min",
        "existing_firms_access_15min", "same_fachgruppe_firms_access_15min",
    ])

print("Input check passed for all required years and columns.")


## 3. Founding model


### 3.1 Load and join inputs


In [ ]:
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

panel_rows = con.execute(
    f"SELECT count(*) FROM read_parquet('{PANEL_PATH.as_posix()}') WHERE year BETWEEN {LAG_YEAR} AND {END_YEAR}"
).fetchone()[0]

founding = con.execute(f"""
SELECT p.grid_id, p.municipality_id, p.year, p.quarter, p.period,
       p.births, p.active_firms_tminus1,
       a.own_cell_pop, a.own_cell_firms,
       a.pop_access_15min, a.existing_firms_access_15min,
       w.own_cell_walk_pop, w.own_cell_walk_firms,
       w.walk_pop_10min, w.walk_firms_10min,
       w.walk_pt_routes_10min, w.pt_ohne_haltestelle,
       n.tt_motorway_exit_min
FROM read_parquet('{PANEL_PATH.as_posix()}') p
LEFT JOIN read_parquet('{(FEATURE_DIR/'*'/'accessibility_potentials_100m.parquet').as_posix()}') a
  USING (grid_id, year, quarter, period)
LEFT JOIN read_parquet('{(FEATURE_DIR/'*'/'pedestrian_accessibility_quarter_100m.parquet').as_posix()}') w
  USING (grid_id, year, quarter, period)
LEFT JOIN read_parquet('{(FEATURE_DIR/'*'/'nearest_infrastructure_100m.parquet').as_posix()}') n
  USING (grid_id, year)
WHERE p.year BETWEEN {LAG_YEAR} AND {END_YEAR}
ORDER BY p.grid_id, p.year, p.quarter
""").df()

if len(founding) != panel_rows:
    raise ValueError("The founding-data joins lost or duplicated panel rows.")
if founding.duplicated(["grid_id", "year", "quarter"]).any():
    raise ValueError("Duplicate cell-quarter rows after the joins.")

print(f"Panel input: {panel_rows:,} rows")
print(f"Joined founding data: {founding.shape[0]:,} rows × {founding.shape[1]} columns")
print(f"Cells: {founding['grid_id'].nunique():,}; quarters: {founding['period'].nunique()}")


### 3.2 Remove incomplete rows


In [ ]:
founding_route_columns = [
    "pop_access_15min", "existing_firms_access_15min", "walk_pop_10min", "walk_firms_10min",
    "walk_pt_routes_10min", "pt_ohne_haltestelle", "tt_motorway_exit_min",
]
incomplete = founding[founding_route_columns].isna().any(axis=1)
print(f"Rows with incomplete routing: {incomplete.sum():,} ({incomplete.mean():.3%})")
founding = founding.loc[~incomplete].copy()
print(f"Rows retained: {len(founding):,}")


### 3.3 Create neighbourhood variables


In [ ]:
# Non-overlapping neighbourhoods. The origin cell is removed.
founding["car_pop_ring_0_15"] = (founding["pop_access_15min"] - founding["own_cell_pop"]).clip(lower=0)
founding["car_firms_ring_0_15"] = (founding["existing_firms_access_15min"] - founding["own_cell_firms"]).clip(lower=0)
founding["walk_pop_ring_0_10"] = (founding["walk_pop_10min"] - founding["own_cell_walk_pop"]).clip(lower=0)
founding["walk_firms_ring_0_10"] = (founding["walk_firms_10min"] - founding["own_cell_walk_firms"]).clip(lower=0)

founding["log_pop_access_ring_0_15"] = np.log1p(founding["car_pop_ring_0_15"])
founding["log_firms_relative_car_ring_0_15"] = (
    np.log1p(founding["car_firms_ring_0_15"]) - founding["log_pop_access_ring_0_15"]
)
founding["log_walk_pop_ring_0_10"] = np.log1p(founding["walk_pop_ring_0_10"])
founding["log_firms_relative_walk_ring_0_10"] = (
    np.log1p(founding["walk_firms_ring_0_10"]) - founding["log_walk_pop_ring_0_10"]
)

print("Created car and walking population-mass and relative-firm-density variables.")


### 3.4 Create lags and growth


In [ ]:
founding = founding.sort_values(["grid_id", "year", "quarter"])
period_number = founding["year"] * 4 + founding["quarter"]
previous_period = period_number.groupby(founding["grid_id"], sort=False).shift(4)
previous_population = founding.groupby("grid_id", sort=False)["own_cell_pop"].shift(4)
previous_population = previous_population.where(period_number - previous_period == 4)

founding["log_own_firms"] = np.log1p(founding["active_firms_tminus1"])
founding["log_own_pop"] = np.log1p(previous_population)
founding["population_growth_yoy"] = np.log1p(founding["own_cell_pop"]) - np.log1p(previous_population)
founding["log_tt_motorway_exit"] = np.log1p(founding["tt_motorway_exit_min"])

before = len(founding)
founding = founding.loc[founding["year"] >= START_YEAR].copy()
print(f"Removed {before-len(founding):,} lead-in rows from {LAG_YEAR}.")


### 3.5 Create the design matrix


In [ ]:
founding_terms = [
    "log_own_firms", "log_own_pop",
    "log_pop_access_ring_0_15", "log_firms_relative_car_ring_0_15",
    "log_walk_pop_ring_0_10", "log_firms_relative_walk_ring_0_10",
    "population_growth_yoy", "log_tt_motorway_exit",
    "walk_pt_routes_10min", "pt_ohne_haltestelle",
]

before = len(founding)
founding = founding.dropna(subset=founding_terms).copy()
period_dummies = pd.get_dummies(founding["period"], prefix="period", drop_first=True, dtype="float64")
X_founding = pd.concat([founding[founding_terms].astype("float64"), period_dummies], axis=1)
X_founding.insert(0, "const", 1.0)
y_founding = founding["births"].astype("float64")
founding_clusters = founding["grid_id"]

if not np.isfinite(X_founding.to_numpy()).all(): raise ValueError("Non-finite founding regressors.")
if (y_founding < 0).any() or not np.allclose(y_founding, np.floor(y_founding)):
    raise ValueError("Births must be non-negative integer counts.")

print(f"Rows removed for missing model values: {before-len(founding):,}")
print(f"Final founding sample: {len(founding):,} rows × {X_founding.shape[1]} design columns")
print(f"Births: {int(y_founding.sum()):,}; zero-count rows: {(y_founding==0).mean():.2%}")
print(f"Cell clusters: {founding_clusters.nunique():,}")


### 3.6 Fit the offset model


In [ ]:
from statsmodels.discrete.discrete_model import NegativeBinomial, Poisson

X_offset = X_founding.drop(columns="log_own_firms")
offset = founding["log_own_firms"].to_numpy()
poisson_offset = Poisson(y_founding, X_offset, offset=offset).fit(maxiter=200, disp=0)
nb_offset = NegativeBinomial(y_founding, X_offset, offset=offset, loglike_method="nb2").fit(
    start_params=np.append(poisson_offset.params.to_numpy(), 0.1), maxiter=500, disp=0
)
print(f"Offset NB2 log-likelihood: {nb_offset.llf:,.2f}")


### 3.7 Fit the unrestricted NB2


In [ ]:
start_values = pd.Series(index=[*X_founding.columns, "alpha"], dtype="float64")
start_values.loc[X_offset.columns] = nb_offset.params.loc[X_offset.columns]
start_values["log_own_firms"] = 1.0
start_values["alpha"] = nb_offset.params["alpha"]

poisson_result = Poisson(y_founding, X_founding).fit(maxiter=200, disp=0)
nb_result = NegativeBinomial(y_founding, X_founding, loglike_method="nb2").fit(
    start_params=start_values.to_numpy(), cov_type="cluster",
    cov_kwds={"groups": founding_clusters}, maxiter=500, disp=1,
)
converged = bool(nb_result.mle_retvals.get("converged", False))
if not converged: raise RuntimeError("The founding NB2 did not converge.")
if nb_result.llf + 1e-6 < nb_offset.llf:
    raise RuntimeError("The unrestricted NB2 has a lower likelihood than its nested offset model.")
lr_statistic = max(0.0, 2*(nb_result.llf-poisson_result.llf))
lr_p_value = 0.5*stats.chi2.sf(lr_statistic, 1)
print(f"Converged: {converged}")
print(f"Unrestricted NB2 log-likelihood: {nb_result.llf:,.2f}")
print(f"Boundary LR test: statistic={lr_statistic:,.2f}, p={lr_p_value:.4g}")


### 3.8 Save founding estimates


In [ ]:
ci = nb_result.conf_int()
founding_results = pd.DataFrame({
    "coefficient":nb_result.params, "std_error":nb_result.bse, "p_value":nb_result.pvalues,
    "ci_lower":ci.iloc[:,0], "ci_upper":ci.iloc[:,1],
})
founding_results["irr"] = np.exp(founding_results["coefficient"])
founding_results["irr_ci_lower"] = np.exp(founding_results["ci_lower"])
founding_results["irr_ci_upper"] = np.exp(founding_results["ci_upper"])
founding_results.loc["alpha", ["irr","irr_ci_lower","irr_ci_upper"]] = np.nan
founding_results.to_csv(RESULT_DIR/"founding_nb2_all_terms.csv", index_label="term")
founding_results.loc[~founding_results.index.str.startswith("period_")].to_csv(
    RESULT_DIR/"founding_nb2_results.csv", index_label="term"
)
founding_diagnostics = pd.Series({
    "n_observations":len(founding), "n_cells":founding.grid_id.nunique(),
    "n_births":int(y_founding.sum()), "zero_share":float((y_founding==0).mean()),
    "alpha":float(nb_result.params["alpha"]), "converged":converged,
    "log_likelihood":float(nb_result.llf), "offset_log_likelihood":float(nb_offset.llf),
    "boundary_lr_statistic":lr_statistic, "boundary_lr_p_value":lr_p_value,
})
founding_diagnostics.to_csv(RESULT_DIR/"founding_nb2_diagnostics.csv", header=["value"])
print(f"Saved founding estimates: {founding_results.shape[0]} terms")


### 3.9 Save founding interpretation tables


In [ ]:
covariance = np.asarray(nb_result.cov_params()); names = list(nb_result.params.index)
def contrast(weights):
    vector=np.zeros(len(names))
    for term,weight in weights.items(): vector[names.index(term)]=weight
    estimate=float(vector@nb_result.params.to_numpy()); se=float(np.sqrt(vector@covariance@vector))
    return estimate,se

rows=[]
for mode,mass,density in [
    ("car","log_pop_access_ring_0_15","log_firms_relative_car_ring_0_15"),
    ("walk","log_walk_pop_ring_0_10","log_firms_relative_walk_ring_0_10"),
]:
    for effect,weights in {
        "population_holding_firms_fixed":{mass:1,density:-1},
        "firms_holding_population_fixed":{density:1},
        "population_and_firms_proportional":{mass:1},
    }.items():
        estimate,se=contrast(weights)
        rows.append({"mode":mode,"effect":effect,"coefficient":estimate,"std_error":se,
                     "irr":np.exp(estimate),"irr_ci_lower":np.exp(estimate-1.96*se),"irr_ci_upper":np.exp(estimate+1.96*se)})
pd.DataFrame(rows).to_csv(RESULT_DIR/"founding_nb2_decomposition.csv",index=False)

mu=nb_result.predict(); alpha=float(nb_result.params["alpha"]); size=1/alpha; probability=size/(size+mu)
counts=[0,1,2,3,4]
modelled=[float(stats.nbinom.pmf(k,size,probability).mean()) for k in counts]
modelled.append(1-sum(modelled))
observed=[float((y_founding==k).mean()) for k in counts]+[float((y_founding>=5).mean())]
pd.DataFrame({"births":[*map(str,counts),"5+"],"observed_share":observed,"modelled_share":modelled}).to_csv(
    RESULT_DIR/"founding_nb2_count_fit.csv",index=False
)
print("Saved decomposition and count-fit tables.")


### 3.10 Prepare the official-sector founding inputs

The sector models use the official `Sparte_ID` mapping. Sectors with fewer than 500 births are omitted because they do not provide enough outcome support for this specification. The three compact cache files are rebuilt from the current inputs on every complete run.


In [ ]:
sector_mapping = (
    pd.read_parquet(FIRMS_PATH, columns=["Fachgruppe_ID", "Sparte_ID", "Sparte_Text"])
    .dropna().astype("string").drop_duplicates()
    .rename(columns={"Sparte_ID": "sparte", "Sparte_Text": "sparte_name"})
)
if sector_mapping.groupby("Fachgruppe_ID")["sparte"].nunique().gt(1).any():
    raise ValueError("A Fachgruppe is assigned to more than one official sector.")
if sector_mapping.groupby("sparte")["sparte_name"].nunique().gt(1).any():
    raise ValueError("An official sector ID has inconsistent names.")

sector_access_path = CACHE_DIR / "sparten_accessibility_100m.parquet"
sector_births_path = CACHE_DIR / "sparten_births_100m.parquet"
sector_stock_path = CACHE_DIR / "sparten_stock_100m.parquet"
for path in [sector_access_path, sector_births_path, sector_stock_path]:
    path.unlink(missing_ok=True)

con.register("sector_mapping", sector_mapping[["Fachgruppe_ID", "sparte"]])
fachgruppe_files = (FEATURE_DIR / "*" / "fachgruppe_accessibility_quarter_100m.parquet" / "**" / "*.parquet").as_posix()
con.execute(f"""
COPY (
    SELECT f.grid_id, f.year, f.quarter, m.sparte,
           sum(f.same_fachgruppe_firms_access_15min) AS same_access_15min,
           sum(f.own_cell_same_fachgruppe_firms) AS own_same
    FROM read_parquet('{fachgruppe_files}', hive_partitioning=true) f
    JOIN sector_mapping m ON CAST(f.Fachgruppe_ID AS VARCHAR)=m.Fachgruppe_ID
    WHERE f.year BETWEEN {LAG_YEAR} AND {END_YEAR}
    GROUP BY 1,2,3,4
) TO '{sector_access_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
con.execute(f"""
COPY (
    SELECT b.grid_id, b.period, m.sparte, sum(b.births) AS births_sparte
    FROM read_parquet('{BIRTHS_BY_GROUP_PATH.as_posix()}') b
    JOIN sector_mapping m ON CAST(b.Fachgruppe_ID AS VARCHAR)=m.Fachgruppe_ID
    GROUP BY 1,2,3
) TO '{sector_births_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

panel_columns = pq.ParquetFile(PANEL_PATH).schema_arrow.names
stock_columns = [name for name in panel_columns if name.startswith("fachgruppe_") and name.endswith("_active_firms_tminus1")]
stock_keys = pd.read_parquet(PANEL_PATH, columns=["grid_id", "period"])
stock_pieces = []
for sector in sorted(sector_mapping["sparte"].unique()):
    groups = set(sector_mapping.loc[sector_mapping["sparte"] == sector, "Fachgruppe_ID"])
    columns = [name for name in stock_columns if name.removeprefix("fachgruppe_").removesuffix("_active_firms_tminus1") in groups]
    if columns:
        piece = stock_keys.copy()
        piece["sparte"] = sector
        piece["own_same_stock"] = pd.read_parquet(PANEL_PATH, columns=columns).sum(axis=1).to_numpy()
        stock_pieces.append(piece)
if not stock_pieces:
    raise ValueError("No sector-specific lagged-stock columns were found.")
pd.concat(stock_pieces, ignore_index=True).to_parquet(sector_stock_path, index=False)

sector_birth_totals = pd.read_parquet(sector_births_path).groupby("sparte")["births_sparte"].sum()
sectors = sorted(str(sector) for sector, births in sector_birth_totals.items() if births >= 500)
print(f"Official sectors in source: {sector_mapping['sparte'].nunique()}")
print(f"Sectors retained (at least 500 births): {len(sectors)}")
print(f"Retained sector IDs: {', '.join(sectors) if sectors else 'none'}")


### 3.11 Build one sector-specific design matrix

The risk set and period indicators remain identical to the main founding model. Firm stocks are separated into the focal official sector and all other sectors.


In [ ]:
def build_sector_design(sector):
    columns = [
        "grid_id", "year", "quarter", "period", "active_firms_tminus1",
        "own_cell_pop", "own_cell_firms", "existing_firms_access_15min",
        "log_own_pop", "log_pop_access_ring_0_15", "log_walk_pop_ring_0_10",
        "population_growth_yoy", "log_tt_motorway_exit",
        "walk_pt_routes_10min", "pt_ohne_haltestelle",
    ]
    frame = founding[columns].copy()
    access = pd.read_parquet(sector_access_path, filters=[("sparte", "=", sector)]).drop(columns="sparte")
    births = pd.read_parquet(sector_births_path, filters=[("sparte", "=", sector)]).drop(columns="sparte")
    stock = pd.read_parquet(sector_stock_path, filters=[("sparte", "=", sector)]).drop(columns="sparte")
    frame = frame.merge(access, on=["grid_id", "year", "quarter"], how="left", validate="one_to_one")
    frame = frame.merge(stock, on=["grid_id", "period"], how="left", validate="one_to_one")
    frame = frame.merge(births, on=["grid_id", "period"], how="left", validate="one_to_one")
    for column in ["same_access_15min", "own_same", "own_same_stock", "births_sparte"]:
        frame[column] = frame[column].fillna(0)

    frame["log_own_same"] = np.log1p(frame["own_same_stock"].clip(lower=0))
    frame["log_own_other"] = np.log1p((frame["active_firms_tminus1"] - frame["own_same_stock"]).clip(lower=0))
    same_ring = (frame["same_access_15min"] - frame["own_same"]).clip(lower=0)
    all_ring = (frame["existing_firms_access_15min"] - frame["own_cell_firms"]).clip(lower=0)
    other_ring = (all_ring - same_ring).clip(lower=0)
    frame["log_same_relative_car_ring_0_15"] = np.log1p(same_ring) - frame["log_pop_access_ring_0_15"]
    frame["log_other_relative_car_ring_0_15"] = np.log1p(other_ring) - frame["log_pop_access_ring_0_15"]

    terms = [
        "log_own_same", "log_own_other", "log_own_pop",
        "log_pop_access_ring_0_15", "log_walk_pop_ring_0_10",
        "log_same_relative_car_ring_0_15", "log_other_relative_car_ring_0_15",
        "population_growth_yoy", "log_tt_motorway_exit",
        "walk_pt_routes_10min", "pt_ohne_haltestelle",
    ]
    period = pd.get_dummies(frame["period"], prefix="period", drop_first=True, dtype="float64")
    design = pd.concat([frame[terms].astype("float64"), period], axis=1)
    design.insert(0, "const", 1.0)
    return design, frame["births_sparte"].astype("float64"), frame["grid_id"], terms

if sectors:
    example_X, example_y, example_clusters, example_terms = build_sector_design(sectors[0])
    print(f"Example sector design: {example_X.shape[0]:,} rows × {example_X.shape[1]} columns")
    print(f"Example sector births: {int(example_y.sum()):,}; cell clusters: {example_clusters.nunique():,}")


### 3.12 Estimate and save the sector models

Each sector NB2 starts from the converged all-sector NB2 estimates. This avoids the overflow seen when a sparse sector-specific Poisson model was used as the only starting point. If BFGS fails, the notebook retries with L-BFGS from the best finite parameters available; it still stops unless the final fit converges with finite cell-clustered standard errors.


In [ ]:
import warnings

sector_result_rows = []
sector_diagnostic_rows = []
sector_population_rows = []
sector_population_influence = {}
for sector in sectors:
    X_sector, y_sector, clusters_sector, terms_sector = build_sector_design(sector)
    model = NegativeBinomial(y_sector, X_sector, loglike_method="nb2")

    start = pd.Series(0.0, index=[*X_sector.columns, "alpha"])
    shared = [name for name in X_sector.columns if name in nb_result.params.index]
    start.loc[shared] = nb_result.params.loc[shared]
    start["log_own_same"] = nb_result.params["log_own_firms"]
    start["log_own_other"] = nb_result.params["log_own_firms"]
    start["log_same_relative_car_ring_0_15"] = nb_result.params["log_firms_relative_car_ring_0_15"]
    start["log_other_relative_car_ring_0_15"] = nb_result.params["log_firms_relative_car_ring_0_15"]
    start["alpha"] = max(float(nb_result.params["alpha"]), 1e-4)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sector_result = model.fit(
            start_params=start.to_numpy(), method="bfgs", maxiter=1000, disp=0,
            cov_type="cluster", cov_kwds={"groups": clusters_sector},
        )

    valid = (bool(sector_result.mle_retvals.get("converged", False))
             and np.isfinite(sector_result.params).all()
             and np.isfinite(sector_result.bse).all()
             and sector_result.params["alpha"] > 0)
    method = "BFGS from all-sector estimates"
    if not valid:
        fallback_start = np.asarray(sector_result.params)
        if not np.isfinite(fallback_start).all():
            fallback_start = start.to_numpy()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            sector_result = model.fit(
                start_params=fallback_start, method="lbfgs", maxiter=2000, disp=0,
                cov_type="cluster", cov_kwds={"groups": clusters_sector},
            )
        valid = (bool(sector_result.mle_retvals.get("converged", False))
                 and np.isfinite(sector_result.params).all()
                 and np.isfinite(sector_result.bse).all()
                 and sector_result.params["alpha"] > 0)
        method = "L-BFGS fallback"
    if not valid:
        raise RuntimeError(f"Official-sector founding model {sector} did not produce a valid converged fit: {sector_result.mle_retvals}")

    sector_name = sector_mapping.loc[sector_mapping["sparte"] == sector, "sparte_name"].iloc[0]
    ci = sector_result.conf_int()
    parameter_names = list(sector_result.params.index)
    population_contrast = np.zeros(len(parameter_names))
    population_contrast[parameter_names.index("log_pop_access_ring_0_15")] = 1
    population_contrast[parameter_names.index("log_same_relative_car_ring_0_15")] = -1
    population_contrast[parameter_names.index("log_other_relative_car_ring_0_15")] = -1
    population_effect = float(population_contrast @ sector_result.params.to_numpy())
    population_se = float(np.sqrt(population_contrast @ np.asarray(sector_result.cov_params()) @ population_contrast))
    sector_population_rows.append({
        "sparte": sector, "sparte_name": sector_name, "coefficient": population_effect,
        "std_error": population_se, "irr": np.exp(population_effect),
        "irr_ci_lower": np.exp(population_effect-1.96*population_se),
        "irr_ci_upper": np.exp(population_effect+1.96*population_se),
    })

    scores = sector_result.model.score_obs(np.asarray(sector_result.params))
    codes, labels = pd.factorize(clusters_sector, sort=False)
    grouped_scores = np.zeros((len(labels), scores.shape[1]))
    np.add.at(grouped_scores, codes, scores)
    bread = -np.linalg.pinv(sector_result.model.hessian(np.asarray(sector_result.params)))
    sector_population_influence[sector] = pd.Series(population_contrast @ bread @ grouped_scores.T, index=labels)

    for term in terms_sector:
        sector_result_rows.append({
            "sparte": sector, "sparte_name": sector_name, "term": term,
            "coefficient": sector_result.params[term], "std_error": sector_result.bse[term],
            "irr": np.exp(sector_result.params[term]),
            "irr_ci_lower": np.exp(ci.loc[term, 0]), "irr_ci_upper": np.exp(ci.loc[term, 1]),
        })
    sector_diagnostic_rows.append({
        "sparte": sector, "sparte_name": sector_name, "n_births": int(y_sector.sum()),
        "alpha": float(sector_result.params["alpha"]), "log_likelihood": float(sector_result.llf),
        "converged": True, "optimizer": method,
    })
    print(f"Sector {sector} ({sector_name}): {int(y_sector.sum()):,} births; converged with {method}")

sector_results = pd.DataFrame(sector_result_rows, columns=["sparte","sparte_name","term","coefficient","std_error","irr","irr_ci_lower","irr_ci_upper"])
sector_diagnostics = pd.DataFrame(sector_diagnostic_rows, columns=["sparte","sparte_name","n_births","alpha","log_likelihood","converged","optimizer"])
sector_population = pd.DataFrame(sector_population_rows, columns=["sparte","sparte_name","coefficient","std_error","irr","irr_ci_lower","irr_ci_upper"])
sector_results.to_csv(RESULT_DIR/"founding_nb2_sector_results.csv", index=False)
sector_diagnostics.to_csv(RESULT_DIR/"founding_nb2_sector_diagnostics.csv", index=False)
sector_population.to_csv(RESULT_DIR/"founding_sector_population_effects.csv", index=False)
print(f"Saved sector estimates: {len(sector_results):,} coefficient rows from {len(sector_diagnostics)} models")


### 3.13 Joint test of the sector population effects


In [ ]:
if len(sector_population) > 1:
    influence = pd.DataFrame(sector_population_influence).fillna(0.0)
    cross_model_covariance = influence.to_numpy().T @ influence.to_numpy()
    estimates = sector_population.set_index("sparte").loc[influence.columns, "coefficient"].to_numpy()
    contrast_matrix = np.column_stack([-np.ones(len(estimates)-1), np.eye(len(estimates)-1)])
    differences = contrast_matrix @ estimates
    difference_covariance = contrast_matrix @ cross_model_covariance @ contrast_matrix.T
    degrees_of_freedom = int(np.linalg.matrix_rank(difference_covariance))
    statistic = float(differences @ np.linalg.pinv(difference_covariance) @ differences)
    joint_test = pd.Series({
        "effect": "reachable_population_holding_same_and_other_firms_fixed",
        "wald_chi2": statistic, "degrees_of_freedom": degrees_of_freedom,
        "p_value": float(stats.chi2.sf(statistic, degrees_of_freedom)),
        "covariance": "cross-model cell-clustered sandwich (CR0)",
    })
    joint_test.to_csv(RESULT_DIR/"founding_sector_joint_test.csv", header=["value"])
    print(f"Joint sector test: chi-square({degrees_of_freedom})={statistic:.2f}, p={joint_test['p_value']:.4g}")
else:
    print("Joint sector test not calculated: fewer than two sector models were estimated.")


## 4. Survival model


### 4.1 Load and join inputs


In [ ]:
firm_feature_glob = (FEATURE_DIR / "*" / "firm_accessibility_quarter_100m.parquet").as_posix()
raw_survival_rows = con.execute(
    f"SELECT count(*) FROM read_parquet('{firm_feature_glob}') WHERE year BETWEEN {START_YEAR} AND {END_YEAR}"
).fetchone()[0]

spells = con.execute(f"""
SELECT a.firm_id AS standort_id, a.grid_id_100m AS grid_id,
       CAST(a.Fachgruppe_ID AS VARCHAR) AS Fachgruppe_ID,
       a.year, a.quarter, a.period, a.included_in_lagged_stock,
       a.own_cell_pop, a.own_cell_firms, a.own_cell_same_fachgruppe_firms,
       a.pop_access_15min, a.existing_firms_access_15min,
       a.same_fachgruppe_firms_access_15min,
       w.walk_pt_routes_10min, w.pt_ohne_haltestelle,
       n.tt_motorway_exit_min
FROM read_parquet('{firm_feature_glob}') a
JOIN read_parquet('{(FEATURE_DIR/'*'/'pedestrian_accessibility_quarter_100m.parquet').as_posix()}') w
  ON a.grid_id_100m=w.grid_id AND a.year=w.year AND a.quarter=w.quarter
JOIN read_parquet('{(FEATURE_DIR/'*'/'nearest_infrastructure_100m.parquet').as_posix()}') n
  ON a.grid_id_100m=n.grid_id AND a.year=n.year
WHERE a.year BETWEEN {START_YEAR} AND {END_YEAR}
ORDER BY a.firm_id, a.year, a.quarter
""").df()

if len(spells) != raw_survival_rows: raise ValueError("The survival joins lost or duplicated rows.")
if spells.duplicated(["standort_id", "year", "quarter"]).any(): raise ValueError("Duplicate firm-quarter rows.")
print(f"Firm-quarter input: {raw_survival_rows:,} rows")
print(f"Joined survival data: {spells.shape[0]:,} rows × {spells.shape[1]} columns")
print(f"Locations: {spells['standort_id'].nunique():,}")


### 4.2 Remove incomplete histories


In [ ]:
survival_route_columns = [
    "pop_access_15min", "existing_firms_access_15min", "same_fachgruppe_firms_access_15min",
    "walk_pt_routes_10min", "pt_ohne_haltestelle", "tt_motorway_exit_min",
]
bad_ids = spells.loc[spells[survival_route_columns].isna().any(axis=1), "standort_id"].unique()
missing_group_ids = spells.loc[spells["Fachgruppe_ID"].isna(), "standort_id"].unique()
excluded_ids = np.union1d(bad_ids, missing_group_ids)
spells = spells.loc[~spells["standort_id"].isin(excluded_ids)].copy()
print(f"Locations removed with incomplete histories: {len(excluded_ids):,}")
print(f"Rows retained: {len(spells):,}; locations retained: {spells['standort_id'].nunique():,}")


### 4.3 Create event intervals


In [ ]:
firm_dates = pd.read_parquet(FIRMS_PATH, columns=[
    "firm_id", "founding_date", "exit_date", "exit_observed", "Sparte_ID", "Sparte_Text",
]).rename(columns={"firm_id":"standort_id", "Sparte_ID":"sparte", "Sparte_Text":"sparte_name"})
if firm_dates["standort_id"].duplicated().any(): raise ValueError("firm_id is not unique.")
firm_dates["founding_date"] = pd.to_datetime(firm_dates["founding_date"], errors="coerce")
firm_dates["exit_date"] = pd.to_datetime(firm_dates["exit_date"], errors="coerce")
firm_dates["sparte"] = firm_dates["sparte"].astype("string")
spells = spells.merge(firm_dates, on="standort_id", how="left", validate="many_to_one")
if spells[["founding_date", "sparte", "sparte_name"]].isna().any().any():
    raise ValueError("A routed location lacks dates or an official sector mapping.")

quarter_index = pd.PeriodIndex(spells["period"], freq="Q").asi8
founding_index = spells["founding_date"].dt.to_period("Q").array.asi8
spells["start"] = quarter_index - founding_index
spells["stop"] = spells["start"] + 1
exit_quarter = spells["exit_date"].dt.to_period("Q").astype("string")
spells["event"] = (spells["exit_observed"].fillna(False) & spells["period"].eq(exit_quarter)).astype("int8")
spells = spells.sort_values(["standort_id", "year", "quarter"])

if (spells["start"] < 0).any(): raise ValueError("A survival interval starts before firm founding.")
if (spells.groupby("standort_id")["event"].sum() > 1).any(): raise ValueError("A location has multiple exits.")
sequence = spells["year"]*4 + spells["quarter"]
if sequence.groupby(spells["standort_id"]).diff().dropna().ne(1).any(): raise ValueError("Firm histories contain gaps.")

entry_age = spells.groupby("standort_id")["start"].min()
print(f"Observed exits: {int(spells['event'].sum()):,}")
print(f"Left-truncated locations: {(entry_age>0).sum():,} ({(entry_age>0).mean():.2%})")


### 4.4 Create covariates


In [ ]:
focal = spells["included_in_lagged_stock"].astype("float64")
spells["own_firms"] = (spells["own_cell_firms"] - focal).clip(lower=0)
spells["own_same"] = (spells["own_cell_same_fachgruppe_firms"] - focal).clip(lower=0)
spells["own_other"] = (spells["own_firms"] - spells["own_same"]).clip(lower=0)

all_firms = (spells["existing_firms_access_15min"] - focal).clip(lower=0)
same_firms = (spells["same_fachgruppe_firms_access_15min"] - focal).clip(lower=0)
other_firms = (all_firms - same_firms).clip(lower=0)
pop_ring = (spells["pop_access_15min"] - spells["own_cell_pop"]).clip(lower=0)
same_ring = (same_firms - spells["own_same"]).clip(lower=0)
other_ring = (other_firms - spells["own_other"]).clip(lower=0)

spells["log_own_pop"] = np.log1p(spells["own_cell_pop"])
spells["log_own_same"] = np.log1p(spells["own_same"])
spells["log_own_other"] = np.log1p(spells["own_other"])
spells["log_pop_ring_0_15"] = np.log1p(pop_ring)
spells["log_same_relative_ring_0_15"] = np.log1p(same_ring) - spells["log_pop_ring_0_15"]
spells["log_other_relative_ring_0_15"] = np.log1p(other_ring) - spells["log_pop_ring_0_15"]
spells["log_tt_motorway_exit"] = np.log1p(spells["tt_motorway_exit_min"])
spells["calendar_year"] = spells["year"] - START_YEAR
print("Removed the focal location and created the Cox-model covariates.")


### 4.5 Create the model frame


In [ ]:
survival_terms = [
    "log_own_pop", "log_own_same", "log_own_other", "log_pop_ring_0_15",
    "log_same_relative_ring_0_15", "log_other_relative_ring_0_15",
    "log_tt_motorway_exit", "walk_pt_routes_10min", "pt_ohne_haltestelle", "calendar_year",
]
survival_frame = spells[[
    "standort_id", "grid_id", "Fachgruppe_ID", "sparte", "sparte_name", "year", "period",
    "start", "stop", "event", *survival_terms,
]].copy()
if not np.isfinite(survival_frame[survival_terms].to_numpy()).all(): raise ValueError("Non-finite Cox covariates.")
print(f"Final survival sample: {survival_frame.shape[0]:,} intervals × {survival_frame.shape[1]} columns")
print(f"Locations: {survival_frame['standort_id'].nunique():,}; exits: {int(survival_frame['event'].sum()):,}")
print(f"Cells: {survival_frame['grid_id'].nunique():,}; Fachgruppe strata: {survival_frame['Fachgruppe_ID'].nunique()}")


### 4.6 Fit the stratified Cox model


In [ ]:
from statsmodels.duration.hazard_regression import PHReg

cox_model = PHReg(
    endog=survival_frame["stop"].to_numpy(), exog=survival_frame[survival_terms].astype("float64"),
    status=survival_frame["event"].to_numpy(), entry=survival_frame["start"].to_numpy(),
    strata=survival_frame["Fachgruppe_ID"].to_numpy(), ties="efron",
)
cox_result = cox_model.fit()
print(f"Cox model fitted with {len(survival_terms)} covariates.")


### 4.7 Calculate cell-clustered standard errors


In [ ]:
score = cox_model.score_residuals(cox_result.params)
rows_outside_risk_sets = np.isnan(score).any(axis=1)
score = np.nan_to_num(score, nan=0.0)
cluster_codes, cluster_labels = pd.factorize(survival_frame["grid_id"], sort=False)
cluster_score = np.zeros((len(cluster_labels), score.shape[1]))
np.add.at(cluster_score, cluster_codes, score)
bread = np.linalg.inv(cox_model.hessian(cox_result.params))
cluster_cov = bread @ (cluster_score.T@cluster_score) @ bread
cluster_bse = np.sqrt(np.clip(np.diag(cluster_cov),0,None))
print(f"Cell clusters: {len(cluster_labels):,}")
print(f"Rows outside all event risk sets: {rows_outside_risk_sets.sum():,} ({rows_outside_risk_sets.mean():.2%})")


### 4.8 Save Cox estimates


In [ ]:
parameter=np.asarray(cox_result.params); critical=stats.norm.ppf(.975); z=parameter/cluster_bse
survival_results=pd.DataFrame({
    "coefficient":parameter,"std_error":cluster_bse,"z":z,"p_value":2*stats.norm.sf(abs(z)),
    "hazard_ratio":np.exp(parameter),"hr_ci_lower":np.exp(parameter-critical*cluster_bse),
    "hr_ci_upper":np.exp(parameter+critical*cluster_bse),
},index=survival_terms)
survival_results.to_csv(RESULT_DIR/"survival_cox_results.csv",index_label="term")
survival_diagnostics=pd.Series({
    "n_intervals":len(survival_frame),"n_locations":survival_frame.standort_id.nunique(),
    "n_events":int(survival_frame.event.sum()),"n_cells":survival_frame.grid_id.nunique(),
    "n_fachgruppe_strata":survival_frame.Fachgruppe_ID.nunique(),
    "rows_outside_event_risk_sets":int(rows_outside_risk_sets.sum()),
})
survival_diagnostics.to_csv(RESULT_DIR/"survival_cox_diagnostics.csv",header=["value"])
print(f"Saved Cox estimates: {survival_results.shape[0]} terms")


### 4.9 Save proportional-hazards diagnostics


In [ ]:
schoenfeld=np.asarray(cox_result.schoenfeld_residuals)
event_rows=~np.isnan(schoenfeld).any(axis=1)
event_age=survival_frame.loc[event_rows,"stop"].to_numpy()/4
ranked_age=stats.rankdata(event_age)
ph_rows=[]
for column,term in enumerate(survival_terms):
    rho,p=stats.spearmanr(ranked_age,schoenfeld[event_rows,column])
    ph_rows.append({"term":term,"spearman_rho":rho,"screen_p_value":p})
pd.DataFrame(ph_rows).to_csv(RESULT_DIR/"survival_schoenfeld_screen.csv",index=False)
print(f"Saved proportional-hazards screen for {len(ph_rows)} covariates.")


### 4.10 Save data needed for descriptive survival figures


In [ ]:
from statsmodels.duration.survfunc import SurvfuncRight
locations=(survival_frame.groupby("standort_id",sort=False)
           .agg(sparte=("sparte","first"),sparte_name=("sparte_name","first"),entry=("start","min"),
                exit=("stop","max"),event=("event","max")).reset_index())
km_rows=[]
for sector,group in locations.groupby("sparte"):
    if group.event.sum()<30: continue
    curve=SurvfuncRight(group.exit.to_numpy()/4,group.event.to_numpy(),entry=group.entry.to_numpy()/4)
    for time,probability in zip(curve.surv_times,curve.surv_prob):
        km_rows.append({"sparte":sector,"sparte_name":group.sparte_name.iloc[0],"time_years":time,
                        "survival_probability":probability,"n_locations":len(group),"n_events":int(group.event.sum())})
pd.DataFrame(km_rows).to_csv(RESULT_DIR/"survival_km_by_sector.csv",index=False)

annual=(survival_frame.groupby(["sparte","sparte_name","year"])
        .agg(exits=("event","sum"),locations_at_risk=("standort_id","nunique")).reset_index())
annual["exit_rate"]=annual.exits/annual.locations_at_risk
annual.to_csv(RESULT_DIR/"survival_annual_exit_rates.csv",index=False)
print(f"Saved {len(km_rows):,} Kaplan–Meier points and {len(annual):,} annual sector rows.")


## 5. Save run information


In [ ]:
manifest={
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
    "study_period":{"start_year":START_YEAR,"end_year":END_YEAR,"founding_lag_year":LAG_YEAR},
    "software":{"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__},
}
(RESULT_DIR/"model_run_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
output_files=sorted(path.name for path in RESULT_DIR.glob("founding_*.csv"))+sorted(path.name for path in RESULT_DIR.glob("survival_*.csv"))+["model_run_manifest.json"]
print(f"Result directory: {RESULT_DIR}")
print(f"Files available for notebook 10: {len(output_files)}")
for name in output_files: print(f"  {name}")
